Group XX | Members: [Student 1], [Student 2] | Question: Q1 | Stage: 3: EDA and data audit

How this works: this notebook audits quality issues without modifying source data, quantifies each issue class, and writes a data quality report generated from code.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from pathlib import Path
from src.config import RAW_FILE, STANDIN_FILE, DOCS_DIR, FIGURE_DIR
from src.cleaning import to_string_df, parse_number, parse_dates
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

raw_path = RAW_FILE if RAW_FILE.exists() else STANDIN_FILE
df = to_string_df(raw_path)
print('raw path', raw_path, 'shape', df.shape)

issues = []
def add_issue(pid, problem, cols, n, sev, reason, treat):
    issues.append({'ID': pid, 'problem': problem, 'columns_affected': cols, 'number_of_records_affected': int(n), 'severity': sev, 'severity_reason': reason, 'proposed_treatment': treat})

# 1 duplicates
norm_farm = df['farm_id'].str.strip().str.lower()
exact_dup = df.duplicated().sum()
near_dup = pd.DataFrame({'farm': norm_farm, 'season': df['season'].str.strip().str.lower(), 'year': parse_dates(df['planting_date']).dt.year}).duplicated(subset=['farm','season','year']).sum()
add_issue(1, 'Exact and near duplicates', 'farm_id, season, planting_date', exact_dup + near_dup, 'High', 'Duplicates distort distributions and leakage in split', 'Remove exact duplicates then keep most complete near-duplicate')

# 2 mixed units
unit_var = df['farm_size_unit'].str.strip().str.lower().nunique()
mixed_unit = (unit_var > 1)
add_issue(2, 'Mixed farm size units', 'farm_size,farm_size_unit', df['farm_size_unit'].notna().sum() if mixed_unit else 0, 'High', 'Incorrect per-hectare normalization risk', 'Convert all sizes to hectares')

# 3 inconsistent category spellings
cat_cols = ['district','province','season','seed_variety','soil_type','pest_incidence','irrigation']
n_incons = 0
for c in cat_cols:
    vals = df[c].astype(str)
    n_incons += (vals.str.strip() != vals).sum() + (vals.str.lower() != vals).sum()
add_issue(3, 'Inconsistent categorical spellings/case', ','.join(cat_cols), n_incons, 'Medium', 'Category fragmentation affects grouping and encoding', 'Normalize strip/case and mapping tables')

# 4 disguised missing
missing_tokens = {'-999','-1','0','N/A','NA','unknown','?','', ' '}
n_disg = 0
for c in df.columns:
    n_disg += df[c].astype(str).str.strip().isin(missing_tokens).sum()
add_issue(4, 'Disguised missing values', 'multiple', n_disg, 'High', 'Silent bad values bias stats', 'Map disguised missing to NaN')

# 5 real missing
n_real_missing = int(df.replace({'':np.nan,' ':np.nan}).isna().sum().sum())
add_issue(5, 'Real missing values', 'multiple', n_real_missing, 'High', 'Missingness can bias model and fairness', 'Impute from train-only statistics')

# 6 mixed date formats/out of range
dt = parse_dates(df['planting_date'])
out_range = ((dt.dt.year < 2023) | (dt.dt.year > 2025)).sum()
add_issue(6, 'Date format/range problems', 'planting_date', int(dt.isna().sum() + out_range), 'High', 'Wrong season alignment and time leakage risk', 'Robust parse + enforce 2023-2025')

# 7 impossible values
alt = parse_number(df['altitude_m']); temp=parse_number(df['avg_temp_c']); ph=parse_number(df['soil_ph'])
rain=parse_number(df['rainfall_mm']); age=parse_number(df['farmer_age']); fert=parse_number(df['fertilizer_kg']); yld=parse_number(df['yield_kg_per_ha'])
n_imp = ((yld<=0)|(ph<4)|(ph>8)|(temp<0)|(temp>60)|(alt<500)|(alt>3500)|(rain<0)|(rain>5000)|(age<18)|(age>100)|(fert<0)).sum()
add_issue(7, 'Impossible/unrealistic values', 'numeric columns', int(n_imp), 'High', 'Violates agronomic/physical constraints', 'Set invalid to NaN, optionally winsorize extremes')

# 8 outliers
num_df = pd.DataFrame({'altitude_m':alt,'temp':temp,'rain':rain,'fert':fert,'yield':yld,'age':age,'soil_ph':ph}).dropna(how='all')
outlier_count=0
for c in num_df.columns:
    s=num_df[c].dropna(); q1,q3=s.quantile([0.25,0.75]); iqr=q3-q1
    outlier_count += ((s < q1-1.5*iqr) | (s > q3+1.5*iqr)).sum()
add_issue(8, 'IQR outliers', 'all numeric', int(outlier_count), 'Medium', 'Some outliers are true extremes; others entry errors', 'Review with domain context and cap only clear errors')

# 9 yield consistency
size = parse_number(df['farm_size'])
unit = df['farm_size_unit'].str.lower().str.strip()
size_ha = size.copy(); size_ha[unit.eq('m2')] = size[unit.eq('m2')]/10000; size_ha[unit.eq('acres')] = size[unit.eq('acres')]*0.404686; size_ha[unit.eq('are')] = size[unit.eq('are')]*0.01
harv = parse_number(df['harvest_weight_kg'])
recalc = harv/size_ha
mismatch = ((yld-recalc).abs()/recalc.abs().clip(lower=1) > 0.1).sum()
add_issue(9, 'Yield mismatch vs harvest/size', 'harvest_weight_kg,farm_size,yield_kg_per_ha', int(mismatch), 'High', 'Target inconsistency hurts supervised learning', 'Recompute when mismatch is large and inputs valid')

# 10 province-district mismatch + coop multi-district
pd_mismatch = (df['province'].str.strip().str.lower().eq('eastern') & df['district'].str.strip().str.lower().eq('huye')).sum()
coop_multi = df.groupby(df['cooperative_id'].str.strip().str.lower())['district'].nunique().gt(1).sum()
add_issue(10, 'Geo hierarchy inconsistencies', 'province,district,cooperative_id', int(pd_mismatch + coop_multi), 'Medium', 'Wrong aggregations by place', 'Use district->province lookup and investigate coop mapping')

# 11 wrong types with units/commas
type_noise = 0
for c in ['farm_size','fertilizer_kg','harvest_weight_kg','yield_kg_per_ha']:
    type_noise += df[c].astype(str).str.contains(',', regex=False).sum() + df[c].astype(str).str.contains('kg', case=False, regex=False).sum()
add_issue(11, 'Numeric fields stored as text/units', 'farm_size,fertilizer_kg,harvest_weight_kg,yield_kg_per_ha', int(type_noise), 'Medium', 'Parsing errors if untreated', 'Strip commas/units then convert types')

# 12 leakage
add_issue(12, 'Post-harvest leakage columns', 'harvest_weight_kg,yield_kg_per_ha', int(df['harvest_weight_kg'].notna().sum()), 'High', 'harvest_weight known after harvest only', 'Exclude from X for risk/forecast tasks')

# 13 irrigation-season conflicts
season = df['season'].str.strip().str.upper()
irr = df['irrigation'].str.strip().str.lower()
conflict = ((season.isin(['A','B'])) & irr.isin(['yes','y','1','true'])).sum()
add_issue(13, 'Irrigation-season conflicts', 'irrigation,season', int(conflict), 'Low', 'Mostly rain-fed in A/B but rare irrigation can exist', 'Review and keep plausible rare values')

# 14 whitespace/case constants high-card IDs
ws = sum((df[c].astype(str) != df[c].astype(str).str.strip()).sum() for c in df.columns)
const = int((df.nunique(dropna=False)<=1).sum())
add_issue(14, 'Whitespace/case, constant cols, high-card IDs', 'multiple', int(ws+const), 'Low', 'Cosmetic noise and identifier complexity', 'Trim text and keep IDs only for split/integrity')

q = pd.DataFrame(issues)
q.to_csv(DOCS_DIR / 'data_quality_report.csv', index=False)
(DOCS_DIR / 'data_quality_report.md').write_text('# Data Quality Report\n\n' + q.to_string(index=False))
print(q)

# plots (>=8)
for c in ['yield_kg_per_ha','fertilizer_kg','rainfall_mm','avg_temp_c']:
    plt.figure(figsize=(6,4)); sns.histplot(parse_number(df[c]), kde=True); plt.title(f'{c} distribution'); plt.tight_layout(); plt.savefig(FIGURE_DIR / f'hist_{c}.png'); plt.close()

plt.figure(figsize=(8,4)); sns.countplot(x=df['season'].astype(str).str.upper()); plt.title('Season counts'); plt.tight_layout(); plt.savefig(FIGURE_DIR / 'bar_season.png'); plt.close()
plt.figure(figsize=(8,4)); sns.boxplot(x=df['seed_variety'].astype(str), y=parse_number(df['yield_kg_per_ha'])); plt.title('Yield by seed variety'); plt.tight_layout(); plt.savefig(FIGURE_DIR / 'box_yield_seed.png'); plt.close()

size_ha_plot = size_ha.replace([np.inf,-np.inf], np.nan)
plt.figure(figsize=(6,4)); sns.scatterplot(x=parse_number(df['fertilizer_kg'])/size_ha_plot, y=parse_number(df['yield_kg_per_ha']), alpha=0.5); plt.title('Yield vs fertilizer/ha'); plt.tight_layout(); plt.savefig(FIGURE_DIR / 'scatter_yield_fertilizer_perha.png'); plt.close()

num_cols = ['altitude_m','fertilizer_kg','rainfall_mm','avg_temp_c','soil_ph','farmer_age','yield_kg_per_ha']
num = pd.DataFrame({c:parse_number(df[c]) for c in num_cols})
plt.figure(figsize=(8,6)); sns.heatmap(num.corr(), cmap='coolwarm', center=0); plt.title('Correlation heatmap'); plt.tight_layout(); plt.savefig(FIGURE_DIR / 'heatmap_corr.png'); plt.close()

# brief interpretation notes
notes = [
    ('hist_yield_kg_per_ha.png', 'Yield shows skew and suspicious extremes, supporting robust cleaning and clipping rules.'),
    ('hist_fertilizer_kg.png', 'Fertilizer totals vary widely because farm sizes differ, motivating fertilizer per hectare.'),
    ('hist_rainfall_mm.png', 'Rainfall has long tails and a few implausible high values likely from entry errors.'),
    ('hist_avg_temp_c.png', 'Temperature centers in agronomic range but includes impossible points needing correction.'),
    ('bar_season.png', 'Seasons A and B dominate records; C is smaller, affecting stratification and fairness checks.'),
    ('box_yield_seed.png', 'Improved/Hybrid seeds tend to higher medians, aligned with stakeholder question B.'),
    ('scatter_yield_fertilizer_perha.png', 'Yield tends to increase with fertilizer/ha but with diminishing and noisy relationship.'),
    ('heatmap_corr.png', 'Numeric correlations are moderate; no single factor explains yield alone.'),
]
(Path(DOCS_DIR / 'eda_chart_interpretations.md')).write_text('# EDA Figure Notes\n\n' + '\n'.join([f'- **{a}**: {b}' for a,b in notes]))


raw path /home/runner/work/ML-Learning/ML-Learning/q1_maize_yield/data/raw/maize_yield_rwanda_SYNTHETIC_STANDIN.csv shape (1220, 20)
    ID                                        problem  \
0    1                      Exact and near duplicates   
1    2                          Mixed farm size units   
2    3        Inconsistent categorical spellings/case   
3    4                       Disguised missing values   
4    5                            Real missing values   
5    6                     Date format/range problems   
6    7                  Impossible/unrealistic values   
7    8                                   IQR outliers   
8    9                 Yield mismatch vs harvest/size   
9   10                  Geo hierarchy inconsistencies   
10  11            Numeric fields stored as text/units   
11  12                   Post-harvest leakage columns   
12  13                    Irrigation-season conflicts   
13  14  Whitespace/case, constant cols, high-card IDs   

           

/home/runner/work/ML-Learning/ML-Learning/src/cleaning.py:78: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/home/runner/work/ML-Learning/ML-Learning/src/cleaning.py:78: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)


944